In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
import time

print("1. Đang đọc và làm sạch dữ liệu (Bao gồm cả Tọa độ địa lý)...")
start_time = time.time()
file_path = '../data/raw/GlobalLandTemperaturesByCity.csv'
df = pd.read_csv(file_path, parse_dates=['dt'], low_memory=False)

# Xóa dòng thiếu Nhiệt độ và Tọa độ
df_clean = df.dropna(subset=['AverageTemperature', 'Latitude', 'Longitude']).copy()

# ---- FEATURE ENGINEERING (Thời gian) ----
df_clean['Year'] = df_clean['dt'].dt.year
df_clean['Month'] = df_clean['dt'].dt.month

def get_season(month):
    if month in [3, 4, 5]: return 'Spring'
    elif month in [6, 7, 8]: return 'Summer'
    elif month in [9, 10, 11]: return 'Autumn'
    else: return 'Winter'
df_clean['Season'] = df_clean['Month'].apply(get_season)
le_season = LabelEncoder()
df_clean['Season_Encoded'] = le_season.fit_transform(df_clean['Season'])

# ---- FEATURE ENGINEERING (Tọa độ) ----
print("2. Đang chuyển đổi Vĩ độ/Kinh độ thành tọa độ số học...")
# Cắt lấy số và chữ hướng (N/S/E/W) để tính toán siêu tốc
lat_val = df_clean['Latitude'].str[:-1].astype(float)
lat_dir = df_clean['Latitude'].str[-1]
df_clean['Lat'] = np.where(lat_dir == 'S', -lat_val, lat_val) # Bán cầu Nam là số âm

lon_val = df_clean['Longitude'].str[:-1].astype(float)
lon_dir = df_clean['Longitude'].str[-1]
df_clean['Lon'] = np.where(lon_dir == 'W', -lon_val, lon_val) # Bán cầu Tây là số âm

# ---- ĐƯA TỌA ĐỘ VÀO MÔ HÌNH ----
features = ['Year', 'Month', 'Season_Encoded', 'Lat', 'Lon'] # Đã thêm Lat, Lon
X = df_clean[features]
y = df_clean['AverageTemperature']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("3. Đang huấn luyện AI thông minh hơn (Vui lòng đợi 1-2 phút)...")
model = HistGradientBoostingRegressor(max_iter=150, random_state=42)
model.fit(X_train, y_train)

print("4. Đang chấm điểm mô hình...")
y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"\n🚀 QUÁ TRÌNH HOÀN TẤT (Mất {time.time() - start_time:.0f} giây)!")
print(f"🔹 Sai số RMSE mới: {rmse:.2f} °C")
print(f"🔹 Độ chính xác R2 mới: {r2:.2f}")

1. Đang đọc và làm sạch dữ liệu (Bao gồm cả Tọa độ địa lý)...
2. Đang chuyển đổi Vĩ độ/Kinh độ thành tọa độ số học...
3. Đang huấn luyện AI thông minh hơn (Vui lòng đợi 1-2 phút)...
4. Đang chấm điểm mô hình...

🚀 QUÁ TRÌNH HOÀN TẤT (Mất 56 giây)!
🔹 Sai số RMSE mới: 1.85 °C
🔹 Độ chính xác R2 mới: 0.97


In [4]:
import joblib
import os

# Tạo thư mục 'model' nếu chưa có
os.makedirs('../model', exist_ok=True)

# Lưu mô hình AI và bộ mã hóa
if 'model' not in globals() or 'le_season' not in globals():
    raise NameError("Biến 'model' hoặc 'le_season' chưa được định nghĩa. Hãy chạy lại ô huấn luyện trước khi lưu.")

joblib.dump(model, '../model/climate_model_v1.pkl')
joblib.dump(le_season, '../model/season_encoder.pkl')

print("Đã lưu mô hình thành công!")

Đã lưu mô hình thành công!
